# [MLP Lab] - Development Cycle of a Deep Neural Network

In this notebook we follow a practical, hands-on workflow for building and training Multi-Layer Perceptrons (MLPs) on the MNIST dataset using PyTorch.

This lab is designed to teach you the **systematic approach** professional deep learning practitioners use when developing models:

1. **Start small**: Begin with a simple model to verify your code works correctly
2. **Scale up**: Increase model capacity until you observe overfitting
3. **Regularize**: Apply techniques like dropout and L2 regularization to improve generalization

All architectures in this lab are MLPs built with `torch.nn.Sequential`, building on concepts from the previous PyTorch framework lab.

## How to Use This Notebook

This notebook is intentionally didactic and structured for learning:

- **Read carefully**: Each section builds on the previous one
- **Run linearly**: Execute cells in order from top to bottom
- **Complete exercises**: Fill in code where you see `pass` or `...`
- **Check your understanding**: Use the questions and checkpoints to verify learning
- **Experiment**: After completing exercises, try changing hyperparameters and observing results

Take your time with each section. Understanding *why* we make certain choices is more important than rushing through.

## Content & Learning Objectives

This lab is divided into 5 main sections, each teaching crucial skills for deep learning development:

### 1️⃣ Data: MNIST and preprocessing
Load and prepare the MNIST dataset with proper train/val/test splits.

> ##### Learning Objectives
> 
> - Load image datasets using `torchvision.datasets`
> - Apply normalization transforms to stabilize training
> - Create proper train/validation/test splits
> - Set up `DataLoader` objects for batched training

### 2️⃣ First model: small MLP + training loop
Build a minimal MLP and implement the complete training pipeline.

> ##### Learning Objectives
>
> - Build MLPs using `nn.Sequential`
> - Count trainable parameters manually and verify with PyTorch
> - Understand why we use `CrossEntropyLoss` for multi-class classification
> - Implement training and evaluation loops from scratch
> - Visualize training curves

### 3️⃣ Increase capacity until overfitting appears
Systematically grow model size to understand the capacity-performance relationship.

> ##### Learning Objectives
>
> - Recognize overfitting by comparing train vs validation metrics
> - Understand the relationship between model capacity and generalization
> - Calculate the generalization gap
> - Make overfitting explicit through experimentation

### 4️⃣ Regularize the over-parameterized model
Apply regularization techniques to improve generalization.

> ##### Learning Objectives
>
> - Implement dropout in `nn.Sequential` models
> - Apply L2 regularization through the optimizer's `weight_decay`
> - Compare different regularization strategies
> - Understand when and why regularization helps

### 5️⃣ Takeaways and best practices
Synthesize everything you've learned into a development workflow.

> ##### Learning Objectives
>
> - Articulate the DNN development cycle
> - Know when to scale up vs regularize
> - Understand trade-offs between different approaches

## Setup code


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms


### Devices in PyTorch

A **device** is where computations happen:

- `cpu`: always available
- `cuda`: NVIDIA GPU acceleration
- `mps`: Apple Silicon GPU acceleration

Model parameters and input tensors must be on the **same device**. Tensors and models can be moved with `.to(device)`, for example:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
X = X.to(device)
y = y.to(device)
```


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


<details>
<summary>Question - why do we move both model and tensors to <code>device</code>?</summary>

PyTorch cannot run operations between tensors on different devices (for example, model weights on GPU and input batch on CPU).
</details>


# 1️⃣ Data: MNIST and preprocessing

## About MNIST

MNIST (Modified National Institute of Standards and Technology) is a classic dataset in machine learning:

- **70,000 grayscale images** of handwritten digits (0-9)
- **28×28 pixels** per image (784 features when flattened)
- **Balanced classes** (roughly equal examples per digit)
- **Split**: 60,000 training + 10,000 test images

We'll further split the training set into:
- **50,000 samples** for training
- **10,000 samples** for validation (to tune hyperparameters)
- **10,000 samples** for testing (final evaluation)

## Why normalize the data?

We apply normalization with MNIST-specific mean (0.1307) and standard deviation (0.3081):

```python
transforms.Normalize((0.1307,), (0.3081,))
```

**Benefits of normalization:**
1. **Stable gradients**: Prevents exploding/vanishing gradients
2. **Faster convergence**: Optimizer can use larger learning rates
3. **Better performance**: Network learns more effectively
4. **Consistent scale**: All features contribute equally

<details>
<summary>Why these specific values (0.1307, 0.3081)?</summary>

These are the empirically computed mean and standard deviation of the entire MNIST training set:
- Mean = 0.1307 (pixel values averaged across all images)
- Std = 0.3081 (standard deviation of pixel values)

Using these values ensures our data has mean ≈ 0 and std ≈ 1, which is ideal for neural network training.
</details>

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

val_size = 10_000
train_size = len(train_full) - val_size
train_dataset, val_dataset = random_split(
    train_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")


### Exercise - create DataLoaders

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 5 minutes on this exercise.
> ```

Create three `DataLoader` objects with the following specifications:

- **Training loader**: `BATCH_SIZE=128`, shuffle the data
- **Validation loader**: `batch_size=256`, don't shuffle (deterministic order)
- **Test loader**: `batch_size=256`, don't shuffle

**Why do we shuffle training data but not validation/test data?**

<details>
<summary>Hint</summary>

Use `DataLoader(dataset, batch_size=..., shuffle=...)` from `torch.utils.data`.

The signature is: `DataLoader(dataset, batch_size, shuffle)`
</details>

<details>
<summary>Why shuffle?</summary>

We shuffle training data to ensure the model sees examples in a random order each epoch, which helps prevent the model from learning spurious patterns based on the order of the data.

We don't shuffle validation/test data because:
1. Evaluation is deterministic regardless of order
2. It makes debugging easier (same order every time)
3. It's slightly more efficient
</details>

In [ ]:
BATCH_SIZE = 128

train_loader = ...  # TODO: Create DataLoader for training
val_loader = ...    # TODO: Create DataLoader for validation
test_loader = ...   # TODO: Create DataLoader for testing

print("DataLoaders ready")

In [ ]:
# Visual sanity check
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 6, figsize=(10, 4))
axes = axes.ravel()
for i in range(12):
    img = images[i].squeeze().numpy()
    img = (img * 0.3081) + 0.1307  # undo normalization for display
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(str(labels[i].item()))
    axes[i].axis("off")

plt.tight_layout()
plt.show()


# 2️⃣ First model: small MLP + training loop

## Building MLPs with `nn.Sequential`

We'll implement helper functions to build MLPs of arbitrary depth:

### `build_deep_mlp(hidden_sizes=[32])`
Creates an MLP with a flexible number of hidden layers.

**Example architectures:**
- `hidden_sizes=[32]` → one hidden layer with 32 units
- `hidden_sizes=[128, 64]` → two hidden layers (128 → 64)
- `hidden_sizes=[256, 128, 64]` → three hidden layers (256 → 128 → 64)

**Architecture pattern:**
```
Input (784) → Flatten → Linear → ReLU → Linear → ReLU → ... → Linear → Output (10)
               ↑                                                         ↑
               Always first                                              Always last
```

### Parameter counting functions

We also implement utilities to understand model capacity:

- **`count_trainable_params(model)`**: Uses PyTorch's automatic counting
- **`show_linear_layer_params(model)`**: Manual layer-by-layer breakdown

These help you verify your understanding of how parameters are calculated.

<details>
<summary>Why do we use nn.Sequential?</summary>

`nn.Sequential` is perfect for simple feed-forward architectures where:
1. Layers execute in order (no branching)
2. Each layer takes output from previous layer
3. No complex control flow needed

For more complex architectures (like ResNets with skip connections), you'd define a custom `nn.Module` class instead.
</details>

In [ ]:
def build_deep_mlp(input_size=28 * 28, hidden_sizes=[32], output_size=10):
    layers = [nn.Flatten()]

    in_size = input_size
    for hidden_size in hidden_sizes:
        layers.append(nn.Linear(in_size, hidden_size))
        layers.append(nn.ReLU())
        in_size = hidden_size

    layers.append(nn.Linear(in_size, output_size))
    model = nn.Sequential(*layers)
    return model


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def show_linear_layer_params(model):
    total = 0
    print("Trainable parameters per Linear layer:")
    for module in model:
        if isinstance(module, nn.Linear):
            params = module.in_features * module.out_features + module.out_features
            print(f"  Linear({module.in_features}, {module.out_features}): {params:,}")
            total += params
    print(f"Total trainable params: {total:,}")


### How trainable parameters are computed

For `Linear(in_features, out_features)`:

`params = in_features * out_features + out_features`

- `in_features * out_features` are weights, i.e. $W \in \mathbb{R}^{\text{out\_features} \times \text{in\_features}}$
- `out_features` are biases, i.e. $b \in \mathbb{R}^{\text{out\_features}}$

Total trainable parameters = sum over all linear layers.


### Exercise - inspect a small architecture

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

Instantiate a model with `hidden_sizes=[32]` and verify your understanding of parameter counting:

1. Create the model using `build_deep_mlp(hidden_sizes=[32])`
2. Print the model architecture
3. Use `show_linear_layer_params()` to see the manual parameter count
4. Compare with `count_trainable_params()` to verify they match

**Questions to check your understanding:**

- How many parameters are in the first `Linear` layer? Calculate it manually.
- How many parameters are in the second `Linear` layer?
- Why do we add `.to(device)` when creating the model?

<details>
<summary>Hint - Parameter counting formula</summary>

For a `Linear(in_features, out_features)` layer:
```
total_params = (in_features × out_features) + out_features
             = weights                      + biases
```

For `Linear(784, 32)`: `(784 × 32) + 32 = 25,088 + 32 = 25,120`
</details>

<details>
<summary>Solution - Why .to(device)?</summary>

PyTorch requires that model parameters and input tensors are on the same device (CPU, CUDA, or MPS). Moving the model to the device ensures all parameters are allocated on that device, avoiding runtime errors when we pass device-specific tensors through the model.
</details>

In [ ]:
small_hidden_sizes = [32]
small_model = ...  # TODO: Create model and move to device

print(small_model)
print("---------------------------------")
show_linear_layer_params(small_model)
print(f"Count from PyTorch: {count_trainable_params(small_model):,}")

## Loss function choice

MNIST is **single-label multiclass** classification (exactly one correct class among 10).
So we use `CrossEntropyLoss`.


### Cross-Entropy vs BCE (important distinction)

It's crucial to understand when to use each loss function:

| Loss Function | Use Case | Target Format | Output Format |
|---------------|----------|---------------|---------------|
| `CrossEntropyLoss` | Multi-class (one correct class) | Class index (0-9) | Logits for each class |
| `BCEWithLogitsLoss` | Binary or multi-label | Binary values (0 or 1) | Single logit or independent logits |

**For MNIST:**
- We have 10 mutually exclusive classes (digits 0-9)
- Each image belongs to exactly ONE class
- Therefore we use `CrossEntropyLoss`

**When would we use BCE?**
- Binary classification (cat vs dog)
- Multi-label classification (image can have multiple tags like "beach", "sunset", "people")

<details>
<summary>Technical note: Why "WithLogits"?</summary>

Both `CrossEntropyLoss` and `BCEWithLogitsLoss` expect **logits** (raw network outputs), not probabilities.

They internally apply the appropriate activation (softmax or sigmoid) combined with the loss computation in a numerically stable way. This avoids numerical instability that can occur when applying activation and loss separately.

**Don't do this:**
```python
probs = torch.softmax(logits, dim=1)  # manual softmax
loss = some_other_loss(probs, target)  # unstable!
```

**Do this:**
```python
loss = nn.CrossEntropyLoss()(logits, target)  # stable!
```
</details>

In [ ]:
criterion = nn.CrossEntropyLoss()
print(criterion)


## Training utilities

In the following exercise, you'll implement the core training infrastructure:

- `train_one_epoch` for gradient updates on training data
- `evaluate` for metric computation on validation/test data (no gradient updates)
- `train_model` to orchestrate the training loop and track per-epoch history

We keep the loops explicit and readable so you can understand exactly what happens at each step.

### Exercise - implement the training loop

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 15-20 minutes on this exercise.
> This is a core exercise for understanding the training loop mechanics.
> ```

**In this exercise, you'll implement three critical functions: `train_one_epoch`, `evaluate`, and `train_model`.**

---

#### Part 1: Understanding `train_one_epoch`

This function performs one complete pass through the training data.

**The standard training loop pattern:**

```python
for each batch in training data:
    1. Zero the gradients       # optimizer.zero_grad()
    2. Forward pass             # logits = model(X_batch)
    3. Compute loss            # loss = criterion(logits, y_batch)
    4. Backward pass           # loss.backward()
    5. Update weights          # optimizer.step()
```

**Why this order matters:**

1. **`zero_grad()` first**: Gradients accumulate by default, so we must zero them
2. **Forward before backward**: We need the loss before computing gradients
3. **Backward before step**: Gradients must be computed before updating weights

**Your task for `train_one_epoch`:**

Implement the following lines:
- Line with the forward pass: compute `logits` by passing `X_batch` through the model
- Line with loss computation: compute `loss` using the criterion
- Line with backward pass: call the backward method on the loss
- Line with optimizer step: update the model weights

**What this function returns:**

- **`avg_loss`**: Average loss across all batches (for monitoring convergence)
- **`avg_acc`**: Average accuracy across all batches (for monitoring performance)

<details>
<summary>Why average by batch_size?</summary>

When computing the average loss for an epoch:

```python
total_loss += loss.item() * batch_size
total_examples += batch_size
avg_loss = total_loss / total_examples
```

We multiply by `batch_size` because `loss.item()` returns the *mean* loss for that batch. To get the total loss across all examples, we multiply by the number of examples in that batch, then divide by the total number of examples at the end.

This ensures the average is correct even if the last batch has a different size.
</details>

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = ...  # TODO: Implement forward pass
        loss = ...    # TODO: Implement loss computation
        ...           # TODO: Implement backward pass
        ...           # TODO: Implement optimizer step

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_examples += batch_size

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

#### Part 2: Understanding `evaluate`

This function computes metrics on validation or test data **without updating model weights**.

**Key differences from training:**

1. **`model.eval()`**: Switches model to evaluation mode
   - Disables dropout (uses all neurons)
   - Fixes batch normalization statistics (if used)
   
2. **No gradient computation**: We don't need gradients for evaluation
   - More memory efficient
   - Faster computation
   
3. **No `optimizer.step()`**: Weights don't change during evaluation

**The evaluation loop pattern:**

```python
model.eval()
with torch.no_grad():  # Disable gradient tracking
    for each batch in validation/test data:
        1. Forward pass        # logits = model(X_batch)
        2. Compute loss        # loss = criterion(logits, y_batch)
        3. Accumulate metrics  # total_loss += ..., total_correct += ...
```

**Your task for `evaluate`:**

Implement the following lines:
- Line with the forward pass: compute `logits` by passing `X_batch` through the model
- Line with loss computation: compute `loss` using the criterion

**Note:** We don't need `torch.no_grad()` context in this implementation because we're not computing gradients by virtue of being in eval mode and not calling `.backward()`.

<details>
<summary>Why model.eval() is critical</summary>

If you forget `model.eval()`:
- **Dropout** will still be active, randomly zeroing activations
- This makes evaluation **stochastic** (different results each time)
- Your validation accuracy will be artificially lower
- Results won't be reproducible

Always use:
- `model.train()` before training
- `model.eval()` before evaluation
</details>

In [ ]:
def evaluate(model, dataloader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = ...  # TODO: Implement forward pass
        loss = ...    # TODO: Implement loss computation

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_examples += batch_size

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

#### Part 3: Understanding `train_model`

This is the high-level training loop over epochs.
Each epoch calls `train_one_epoch` and then `evaluate`, and stores all metrics in a `history` dictionary.

**Your task for `train_model`:**

Implement the following lines:
- Line calling `train_one_epoch`: get training loss and accuracy for the epoch
- Line calling `evaluate`: get validation loss and accuracy for the epoch
- Lines appending metrics to history: store train_loss, train_acc, val_loss, and val_acc

**What this function does:**

1. Initializes a `history` dictionary to store metrics over time
2. For each epoch:
   - Trains for one epoch and gets training metrics
   - Evaluates on validation set and gets validation metrics
   - Stores all metrics in the history dictionary
   - Optionally prints progress (if `verbose=True`)
3. Returns the complete training history

This history is essential for:
- Plotting learning curves
- Diagnosing overfitting/underfitting
- Comparing different models and hyperparameters

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5, verbose=True):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = ...  # TODO: Call train_one_epoch
        val_loss, val_acc = ...      # TODO: Call evaluate

        ...  # TODO: Append train_loss to history
        ...  # TODO: Append train_acc to history
        ...  # TODO: Append val_loss to history
        ...  # TODO: Append val_acc to history

        if verbose:
            print(
                f"Epoch {epoch:02d}/{epochs} | "
                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
            )

    return history

---

#### Part 4: Visualization helper

The `plot_history` function below visualizes loss and accuracy curves for train and validation sets.
It is useful to diagnose underfitting and overfitting patterns.

This is provided for you - no implementation required.

In [ ]:
def plot_history(history, title="Training curves"):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
    axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], marker="o", label="Train")
    axes[1].plot(epochs, history["val_acc"], marker="o", label="Validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


### `train_and_summarize`

This utility wraps an entire experiment:

- build model (with or without dropout)
- train for several epochs
- evaluate on test set
- return a compact summary row for comparisons

This is what we use later for capacity and regularization studies.


In [ ]:
def train_and_summarize(
    name,
    hidden_sizes,
    train_loader,
    val_loader,
    test_loader,
    epochs=8,
    lr=1e-3,
    seed=42,
    verbose=False,
):
    set_seed(seed)

    model = build_deep_mlp(hidden_sizes=hidden_sizes).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        epochs=epochs,
        verbose=verbose,
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion)

    summary = {
        "name": name,
        "hidden_sizes": tuple(hidden_sizes),
        "params": count_trainable_params(model),
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "generalization_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "test_acc": test_acc,
    }

    return model, history, summary


### Exercise - train the first small model

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Use the `train_and_summarize` helper function to train your first small model.

**Your task:**
1. Train a model with `hidden_sizes=[32]` for 5 epochs
2. Set `lr=1e-3` (learning rate)
3. Set `weight_decay=0.0` (no L2 regularization yet)
4. Enable `verbose=True` to see training progress
5. Plot the training history
6. Examine the summary DataFrame

**What to observe:**
- Are train and validation accuracy close together or far apart?
- Is the model still improving at epoch 5, or has it plateaued?
- What's the final test accuracy?

<details>
<summary>Hint</summary>

```python
small_model, small_history, small_summary = train_and_summarize(
    name="small_mlp",
    hidden_sizes=[32],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=5,
    lr=...,           # fill this in
    weight_decay=..., # fill this in
    verbose=...,      # fill this in
)
```
</details>

<details>
<summary>Interpretation checkpoint</summary>

With only 32 hidden units, this model is relatively small. You should observe:
- Training and validation curves that are close together (low generalization gap)
- Both curves still improving at epoch 5 (the model could benefit from more training)
- Moderate accuracy (~95-97% on MNIST)

This indicates the model is **underfitting** - it has room to improve with either:
1. More training epochs
2. Higher capacity (more/larger hidden layers)
</details>

In [ ]:
small_model, small_history, small_summary = train_and_summarize(
    name="small_mlp",
    hidden_sizes=small_hidden_sizes,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=10,
    lr=...,      # TODO: Fill in learning rate
    verbose=..., # TODO: Fill in verbose flag
)

plot_history(small_history, title="Small MLP [32] (no dropout)")
pd.DataFrame([small_summary])

# 3️⃣ Increase capacity until overfitting appears

## Why reduce the training set size?

To make overfitting more apparent, we **intentionally create a challenging scenario**:

- Original training set: 50,000 samples
- Reduced training set: **2,500 samples** (5% of data)

**Why does this help us learn?**

With less training data:
1. **Overfitting happens faster**: Large models memorize limited data quickly
2. **The gap is more visible**: Train accuracy climbs while validation stalls
3. **Pedagogical value**: You'll clearly see the capacity-generalization trade-off

**In real projects**, you'd use all available training data. This is a learning exercise to make the effect obvious.

<details>
<summary>What would happen with the full 50k samples?</summary>

With 50,000 training samples:
- Models would need to be *much* larger to overfit
- Training would take longer
- The overfitting signal would be weaker and harder to observe
- You might not see clear overfitting even with large models

By using only 5,000 samples, we create an "easy to overfit" scenario that clearly demonstrates the concepts.
</details>

In [ ]:
OVERFIT_TRAIN_SAMPLES = 2500

overfit_train_dataset = Subset(train_dataset, range(OVERFIT_TRAIN_SAMPLES))
overfit_train_loader = DataLoader(overfit_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Subset for overfitting study: {len(overfit_train_dataset):,} samples")


### Exercise - capacity sweep (no regularization)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 15-20 minutes on this exercise.
> This exercise is crucial for understanding the capacity-generalization trade-off.
> ```

Now we'll systematically compare models of different sizes to understand how capacity affects performance.

**Your task:**

Train the following four architectures on the reduced dataset (`overfit_train_loader` with only 5,000 samples):

1. `[32]` - single hidden layer, 32 units
2. `[128, 64]` - two hidden layers
3. `[256, 128, 64]` - three hidden layers  
4. `[512, 256, 128, 64]` - four hidden layers (largest)

For each architecture:
- Train for 8 epochs with `lr=1e-3` and `weight_decay=0.0`
- Collect the summary in a list
- Compare final train accuracy, validation accuracy, and generalization gap

**Key questions:**
- Which model has the highest training accuracy?
- Which model has the highest validation accuracy?
- Which model has the largest generalization gap?
- What does this tell you about model capacity and overfitting?

<details>
<summary>Hint - Loop structure</summary>

```python
capacity_rows = []
for hidden_sizes in capacity_architectures:
    name = f"mlp_{'_'.join(map(str, hidden_sizes))}"
    _, _, summary = train_and_summarize(
        name=name,
        hidden_sizes=hidden_sizes,
        train_loader=overfit_train_loader,  # note: reduced dataset!
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=8,
        lr=1e-3,
        weight_decay=0.0,
        verbose=False,
    )
    capacity_rows.append(summary)
```
</details>

<details>
<summary>Expected observation</summary>

As model capacity increases:
- **Training accuracy** should increase (larger models can fit the training data better)
- **Validation accuracy** may improve initially, then plateau or decrease
- **Generalization gap** (train_acc - val_acc) should widen significantly

This demonstrates the fundamental trade-off: more capacity helps fit the training data, but can hurt generalization if we don't use regularization.
</details>

In [ ]:
capacity_architectures = [
    [32],
    [128, 64],
    [256, 128, 64],
    [512, 256, 128, 64],
]

capacity_rows = []
for hidden_sizes in capacity_architectures:
    name = f"mlp_{'_'.join(map(str, hidden_sizes))}"
    _, _, summary = ...  # TODO: Call train_and_summarize with appropriate parameters
    capacity_rows.append(summary)

capacity_df = pd.DataFrame(capacity_rows).sort_values("params").reset_index(drop=True)
capacity_df[["name", "hidden_sizes", "params", "final_train_acc", "final_val_acc", "generalization_gap", "test_acc"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(capacity_df["params"], capacity_df["final_train_acc"], marker="o", label="Train")
axes[0].plot(capacity_df["params"], capacity_df["final_val_acc"], marker="o", label="Validation")
axes[0].set_xscale("log")
axes[0].set_xlabel("Trainable parameters (log scale)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Capacity vs accuracy")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].bar(capacity_df["name"], capacity_df["generalization_gap"])
axes[1].set_ylabel("Train acc - Val acc")
axes[1].set_title("Generalization gap")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


<details>
<summary>Interpretation checkpoint</summary>

If larger models improve train accuracy much faster than validation accuracy, the gap widens.
That widening gap is the overfitting signal we were looking for.
</details>


### Make overfitting explicit on the largest architecture

Train the largest model longer, without regularization.


In [ ]:
largest_hidden_sizes = [512, 256, 128, 64]

large_model, large_history, large_summary = train_and_summarize(
    name="large_no_reg",
    hidden_sizes=largest_hidden_sizes,
    train_loader=overfit_train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=15,
    lr=1e-3,
    verbose=False,
)

plot_history(large_history, title="Largest MLP (no regularization)")
pd.DataFrame([large_summary])


# 4️⃣ Regularize the over-parameterized model

Now we keep the largest architecture fixed and test regularization strategies.


## Regularization techniques

When a model overfits (high training accuracy, lower validation accuracy), we need **regularization** to improve generalization.

### Two main regularization approaches:

#### 1. Dropout (`nn.Dropout`)

**What it does:**
- During training, randomly sets a fraction of activations to zero
- Each forward pass uses a different random subset of the network
- Forces the network to learn redundant representations
- At test time, uses all units but scales outputs appropriately

**How to use:**
```python
nn.Dropout(p=0.3)  # drops 30% of activations during training
```

**Common dropout values:** 0.2 to 0.5 (0.3 is a good starting point)

#### 2. L2 Regularization (Weight Decay)

**What it does:**
- Adds a penalty term to the loss: $L_{total} = L_{data} + \lambda \sum w_i^2$
- Encourages weights to be small
- Prevents any single weight from having too much influence
- Implemented via the `weight_decay` parameter in optimizers

**How to use:**
```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
```

**Common weight decay values:** 1e-5 to 1e-3 (1e-4 is a good starting point)

#### 3. Combining both

Dropout and L2 regularization work through different mechanisms, so they can be used together for stronger regularization:
- Dropout reduces co-adaptation between neurons
- L2 penalizes large weights

<details>
<summary>When to use which?</summary>

**Use dropout when:**
- You have a large, deep network
- Your network has many parameters
- You want to prevent co-adaptation between features

**Use L2 when:**
- You want to prevent weight explosion
- You prefer simpler, smoother decision boundaries
- You want a simpler, more interpretable approach

**Use both when:**
- You have severe overfitting
- You have limited training data
- You want the strongest regularization effect

**Start with:** Try each individually first, then combine if needed.
</details>

### Exercise - add dropout support and compare regularization variants

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 20-25 minutes on this exercise.
> This is the capstone exercise that brings together everything you've learned.
> ```

This exercise has two parts:

#### Part 1: Implement `build_deep_mlp_with_dropout`

The existing `build_deep_mlp` function doesn't support dropout. Create a new function `build_deep_mlp_with_dropout` that:
- Takes a `dropout` parameter (probability of dropping a unit)
- Adds `nn.Dropout(dropout)` after each ReLU activation
- Does NOT add dropout after the final output layer

<details>
<summary>Hint - Where to add dropout</summary>

```python
def build_deep_mlp_with_dropout(input_size=28 * 28, hidden_sizes=[32], output_size=10, dropout=0.3):
    layers = [nn.Flatten()]
    
    in_size = input_size
    for hidden_size in hidden_sizes:
        layers.append(nn.Linear(in_size, hidden_size))
        layers.append(nn.ReLU())
        layers.append(...)  # Add dropout here!
        in_size = hidden_size
    
    layers.append(nn.Linear(in_size, output_size))
    model = nn.Sequential(*layers)
    return model
```
</details>

#### Part 2: Update `train_and_summarize` to support dropout and weight_decay

Modify the `train_and_summarize` function to:
- Accept `dropout` and `weight_decay` parameters
- Use `build_deep_mlp_with_dropout` when `dropout > 0`
- Use the original `build_deep_mlp` when `dropout == 0`
- Pass `weight_decay` to the optimizer

<details>
<summary>Hint - Conditional model creation</summary>

```python
if dropout > 0:
    model = build_deep_mlp_with_dropout(hidden_sizes=hidden_sizes, dropout=dropout).to(device)
else:
    model = build_deep_mlp(hidden_sizes=hidden_sizes).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
```
</details>

#### Part 3: Compare regularization strategies

Once you've updated the functions, compare these four configurations on the largest model `[512, 256, 128, 64]`:

1. No regularization: `dropout=0.0, weight_decay=0.0`
2. Dropout only: `dropout=0.3, weight_decay=0.0`
3. L2 only: `dropout=0.0, weight_decay=1e-4`
4. Both: `dropout=0.3, weight_decay=1e-4`

**Questions to answer:**
- Which configuration achieves the best validation accuracy?
- Which configuration has the smallest generalization gap?
- Does combining dropout + L2 help more than either alone?

In [ ]:
def build_deep_mlp_with_dropout(input_size=28 * 28, hidden_sizes=[32], output_size=10, dropout=0.3):
    layers = [nn.Flatten()]

    in_size = input_size
    for hidden_size in hidden_sizes:
        layers.append(nn.Linear(in_size, hidden_size))
        layers.append(nn.ReLU())
        ...  # TODO: Add dropout layer here
        in_size = hidden_size

    layers.append(nn.Linear(in_size, output_size))
    model = nn.Sequential(*layers)
    return model

#### Now implement the updated `train_and_summarize` function

Below is the template for the updated function. Fill in the missing parts marked with `...` or `pass`.

**Key changes to make:**
1. Add `dropout=0.0` and `weight_decay=0.0` as parameters
2. Choose which model builder to use based on the dropout parameter
3. Pass `weight_decay` to the optimizer
4. Include dropout and weight_decay in the summary dictionary

In [ ]:
def train_and_summarize(
    name,
    hidden_sizes,
    train_loader,
    val_loader,
    test_loader,
    epochs=8,
    lr=1e-3,
    dropout=0.0,
    weight_decay=0.0,
    seed=42,
    verbose=False,
):
    set_seed(seed)

    # TODO: Choose which model builder to use based on dropout parameter
    if ...:
        model = ...
    else:
        model = ...

    # TODO: Create optimizer with weight_decay parameter
    optimizer = ...

    history = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        epochs=epochs,
        verbose=verbose,
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion)

    summary = {
        "name": name,
        "hidden_sizes": tuple(hidden_sizes),
        "params": count_trainable_params(model),
        "dropout": dropout,
        "weight_decay": weight_decay,
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "generalization_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "test_acc": test_acc,
    }

    return model, history, summary

In [ ]:
regularization_settings = [
    ("no_reg", 0.0, 0.0),
    ("dropout_0.3", 0.3, 0.0),
    ("l2_1e-4", 0.0, 1e-4),
    ("dropout_0.3_plus_l2", 0.3, 1e-4),
]

reg_rows = []
for name, dropout, weight_decay in regularization_settings:
    _, _, summary = train_and_summarize(
        name=name,
        hidden_sizes=largest_hidden_sizes,
        train_loader=overfit_train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=15,
        lr=1e-3,
        dropout=dropout,
        weight_decay=weight_decay,
        verbose=False,
    )
    reg_rows.append(summary)

reg_df = pd.DataFrame(reg_rows).sort_values("test_acc", ascending=False).reset_index(drop=True)
reg_df[["name", "params", "dropout", "weight_decay", "final_train_acc", "final_val_acc", "generalization_gap", "test_acc"]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(len(reg_df))
width = 0.35

axes[0].bar(x - width / 2, reg_df["final_val_acc"], width=width, label="Validation")
axes[0].bar(x + width / 2, reg_df["test_acc"], width=width, label="Test")
axes[0].set_xticks(x)
axes[0].set_xticklabels(reg_df["name"], rotation=15)
axes[0].set_ylim(0.85, 1.00)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Regularization comparison")
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

axes[1].bar(reg_df["name"], reg_df["generalization_gap"])
axes[1].set_ylabel("Train acc - Val acc")
axes[1].set_title("Generalization gap after regularization")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


# 5️⃣ Summary & Best Practices

## The DNN Development Cycle

This notebook demonstrated a systematic workflow that you should follow for any deep learning project:

### Step 1: Start Simple ✅
- Build a minimal model first (e.g., single hidden layer with few units)
- **Why?** Verify your data pipeline, training loop, and evaluation code work correctly
- **What to check:** Model trains without errors, loss decreases, accuracy improves

### Step 2: Verify Understanding 🔍
- Count parameters manually and verify with PyTorch
- Visualize training curves (loss and accuracy)
- Check that train and validation metrics make sense
- **Why?** Catch bugs early before scaling up

### Step 3: Scale Up Capacity 📈
- Gradually increase model size (more layers, more units)
- Monitor the generalization gap (train_acc - val_acc)
- **Why?** Find the capacity sweet spot for your problem
- **Red flag:** If train_acc >> val_acc, you're overfitting

### Step 4: Apply Regularization 🛡️
- When overfitting appears, add dropout and/or L2 regularization
- Try each technique individually first, then combine
- **Why?** Control overfitting while maintaining capacity
- **What to check:** Generalization gap should decrease

### Step 5: Iterate and Refine 🔄
- Adjust hyperparameters (learning rate, dropout rate, weight decay)
- Monitor test accuracy as your final benchmark
- **Why?** Optimize for the best generalization performance

## Key Takeaways

1. **Model capacity is a double-edged sword**
   - Too little → underfitting (poor train AND val accuracy)
   - Too much → overfitting (good train, poor val accuracy)
   - Just right → good generalization (good train AND val accuracy)

2. **The generalization gap is your guide**
   - Small gap → model generalizes well, can potentially scale up
   - Large gap → overfitting, need regularization
   - Growing gap → early warning sign during training

3. **Start simple, scale up systematically**
   - Don't jump to huge models immediately
   - Verify each step works before adding complexity
   - This saves debugging time and computational resources

4. **Use the right loss function**
   - Multi-class classification → `CrossEntropyLoss`
   - Binary classification → `BCEWithLogitsLoss`
   - Always use "WithLogits" versions for numerical stability

5. **Regularization techniques serve different purposes**
   - Dropout → prevents co-adaptation, forces redundancy
   - L2 (weight decay) → prevents large weights, smoother functions
   - Combined → strongest regularization for severe overfitting

## Common Pitfalls to Avoid

❌ **Don't:**
- Start with a huge model without baseline
- Ignore the validation set (training accuracy alone is meaningless)
- Apply heavy regularization to an underfitting model
- Use test set for hyperparameter tuning (that's what validation is for!)
- Forget to move tensors to the right device

✅ **Do:**
- Plot training curves every time
- Compare train vs validation metrics
- Understand *why* you're making each change
- Keep validation and test sets separate
- Document what works and what doesn't

## Optional Extensions & Further Exploration

If you've completed all exercises and want to deepen your understanding, try these challenges:

### 🔥 Challenge 1: Extreme Capacity
- Build much larger models: `[1024, 512, 256, 128]` or even deeper
- How large does the generalization gap get?
- Can regularization still help at extreme capacity?

### 🔥 Challenge 2: Regularization Tuning
- Try different dropout rates: 0.1, 0.2, 0.3, 0.4, 0.5
- Try different weight decay values: 1e-5, 1e-4, 1e-3, 1e-2
- Create a heatmap of validation accuracy vs (dropout, weight_decay)

### 🔥 Challenge 3: Training Dynamics
- Train for many more epochs (e.g., 50 or 100)
- Plot how the generalization gap evolves over time
- Does overfitting get worse, or does it plateau?

### 🔥 Challenge 4: Other Regularization Techniques
Research and implement:
- **Batch Normalization** (`nn.BatchNorm1d`)
- **Early Stopping** (stop when validation loss stops improving)
- **Learning Rate Scheduling** (reduce LR when plateauing)

### 🔥 Challenge 5: Different Datasets
Apply the same development cycle to:
- Fashion-MNIST (same format as MNIST, harder problem)
- CIFAR-10 (color images, 10 classes)
- Does the same capacity/regularization strategy work?

### 🔥 Challenge 6: Ablation Study
- What happens without `Flatten()`?
- What if you use `Tanh` instead of `ReLU`?
- What if you don't normalize the input data?
- Document your findings!

---

**Remember:** The goal is not just to achieve high accuracy, but to **understand** the relationship between model capacity, regularization, and generalization. These insights will serve you in all future deep learning projects!